# Comparative Legal Clause Classification

This notebook builds a reproducible NLP coursework pipeline for legal clause classification using LEDGAR as the main dataset. The input is a legal clause or provision text, and the output is a predicted clause category.

The notebook compares four families of approaches:

- dummy baselines for lower-bound context
- classical sparse-text models using TF-IDF features
- an optional fine-tuned transformer classifier
- an optional Qwen2.5-Instruct prompting baseline

A small human-in-the-loop review prototype is included only as an illustrative extension. It is not a legal advice system and does not claim to assess legal risk.

The implementation lives in `modules/`, while this notebook acts as the orchestration, reporting, and display layer. Results are generated only by running the cells; no metrics are inserted manually.


## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook. The project root must contain `pyproject.toml`, `modules/`, `notebooks/`, and `requirements-colab.txt`.

Governance and reproducibility controls in this stage:

| Setting | Value | Purpose |
|---|---:|---|
| `SEED` | `42` | Makes sampling, baseline randomness, and train/test helper behavior reproducible. |
| `DATASET_NAME` | `LEDGAR` | Keeps the main experiment scoped to LEDGAR clause classification. |
| `TOP_K_LABELS` | `20` | Restricts the task to the 20 most frequent training labels for a manageable coursework experiment. |
| `RUN_CLASSICAL_MODELS` | `True` | Enables TF-IDF model experiments. |
| `RUN_TRANSFORMER` | `True` | Attempts transformer fine-tuning only when the runtime can support it. |
| `RUN_QWEN_BASELINE` | `True` | Attempts Qwen prompting only when GPU/model loading is available. |
| `RUN_AGENTIC_EXTENSION` | `True` | Enables a small review workflow demonstration, not an autonomous agent. |

Model and feature hyperparameters declared here:

| Component | Hyperparameters |
|---|---|
| TF-IDF search | `max_features` in `[10000, 30000]`; `ngram_range` in `[(1, 1), (1, 2)]`; `lowercase=True`; `stop_words=None` |
| Transformer | `distilbert-base-uncased`; `max_length=256` |
| Optional legal transformer | `nlpaueb/legal-bert-base-uncased` can be substituted manually if GPU resources allow |
| Qwen prompting | `Qwen/Qwen2.5-3B-Instruct`; test sample size `200`; one few-shot example per class when available |

Explainability note: keeping all configuration values in one cell makes it clear which choices affect runtime cost, model capacity, and evaluation scope.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Colab users: leave this blank for auto-detection, or set it to the
# folder that contains pyproject.toml, modules/, and requirements-colab.txt.
# Example: "/content/drive/MyDrive/NLP/Natural-Language-Processing"
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/Natural-Language-Processing"),
        Path("/content/DLIA/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/DLIA/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/NLP/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/Education/NLP/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/GitHub/Education/NLP/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/Colab Notebooks/Natural-Language-Processing"),
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )


PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]


def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"

if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)


import numpy as np
import pandas as pd
from IPython.display import display

from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import create_ledgar_eda, preprocess_ledgar
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.transformer_model import train_transformer_classifier
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis

SEED = 42
DATASET_NAME = "LEDGAR"
TOP_K_LABELS = 20
MAX_FEATURES_LIST = [10000, 30000]
NGRAM_RANGES = [(1, 1), (1, 2)]
RUN_CLASSICAL_MODELS = True
RUN_TRANSFORMER = True
RUN_QWEN_BASELINE = True
RUN_AGENTIC_EXTENSION = True
RUN_NAIVE_BAYES = True
TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"
OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_TRANSFORMER_LENGTH = 256
QWEN_EVAL_SAMPLE_SIZE = 200
QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1

DOWNLOAD_LEDGAR_IF_MISSING = True
DOWNLOAD_CUAD_IF_MISSING = True
USE_HF_CACHE = True
FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)

print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")
try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")
except Exception:
    print("PyTorch is unavailable. Transformer/Qwen sections will skip if they require it.")


Google Colab runtime detected.
Google Drive is already available.
Project root: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing
Colab runtime: True
Raw LEDGAR directory: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/data/raw/lexglue_ledgar
Results directory: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/results
Device: cuda
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Dataset Download and Raw Setup

This stage obtains the raw datasets without training or preprocessing models. LEDGAR remains the main classification dataset. CUAD is treated separately because it is structured as a contract-review question-answering/span-extraction dataset rather than a direct clause-classification dataset.

Data governance choices:

- LEDGAR is loaded from Hugging Face with `load_dataset("coastalcph/lex_glue", "ledgar")` when local JSONL files are missing.
- Official LEDGAR train, validation, and test splits are preserved when available.
- Raw LEDGAR split exports are saved under `data/raw/lexglue_ledgar/` as JSONL files.
- CUAD raw files are downloaded from `theatticusproject/cuad` when available, but CUAD is not merged with LEDGAR.
- If CUAD is missing, the notebook prints a clear message and continues with LEDGAR.

Inputs and outputs:

| Input | Output |
|---|---|
| Hugging Face LEDGAR or local JSONL | `ledgar_raw_splits` dictionary with train/validation/test DataFrames |
| Optional CUAD raw files | `cuad_clause_df` containing extracted span-level examples for optional inspection |

This stage deliberately does not select labels, encode classes, train models, or compute metrics.


In [ ]:
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)
cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)
raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

Loading LEDGAR from data/raw/lexglue_ledgar JSONL files.
LEDGAR train: 60000 rows, columns=['text', 'label']
LEDGAR validation: 10000 rows, columns=['text', 'label']
LEDGAR test: 10000 rows, columns=['text', 'label']
Using existing CUAD files from data/raw/cuad/.

Dataset availability:
- LEDGAR downloaded/loaded: yes
- LEDGAR train size: 60000
- LEDGAR validation size: 10000
- LEDGAR test size: 10000
- CUAD JSON found: yes
- CUAD master clauses CSV found: yes
- CUAD adapted span examples available for optional analysis: 13062


## 3. LEDGAR Preprocessing and EDA

This stage converts raw LEDGAR into a consistent clause-classification schema used by all later experiments.

Preprocessing decisions:

- Standard schema: `text`, `label`, `label_id`, `split`, `source_dataset`.
- Text cleaning is intentionally light: whitespace is normalised and leading/trailing spaces are stripped.
- Legal punctuation and stopwords are retained because they may carry meaning in contractual language.
- Empty or malformed examples are removed.
- Exact duplicate `text` plus `label` pairs are removed to reduce repeated rows.
- The top `TOP_K_LABELS=20` labels are selected using the training split only, preventing validation/test leakage in label selection.
- Label IDs are assigned after filtering so every model uses the same `label2id` and `id2label` mappings.

EDA outputs generated here:

| Output | Purpose |
|---|---|
| `class_distribution.png` | Shows imbalance across the selected labels. |
| `clause_length_histogram.png` | Shows clause length variation, useful for interpreting transformer truncation risk. |
| `dataset_split_summary.csv` | Records split sizes and class counts. |
| `examples_per_label.jsonl` | Provides qualitative examples for annotation and ambiguity inspection. |

The processed JSONL files are saved under `data/processed/` and become the controlled inputs for all model sections.


In [ ]:
processed_splits, label2id, id2label = preprocess_ledgar(
    ledgar_raw_splits,
    paths,
    top_k_labels=TOP_K_LABELS,
    dataset_name=DATASET_NAME,
)
split_summary = create_ledgar_eda(processed_splits, paths.results_dir)

if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))
else:
    train_df = validation_df = test_df = pd.DataFrame(columns=["text", "label", "label_id", "split", "source_dataset"])
    label_names = []
    print("Main LEDGAR experiment cannot run without LEDGAR data.")

,split,rows,classes
0,train,28587,20
1,validation,4670,20
2,test,4732,20


,label
0,Governing Laws
1,Notices
2,Counterparts
3,Entire Agreements
4,Severability
5,Amendments
6,Survival
7,Assignments
8,Expenses
9,Terms


## 4. Shared Result State

This short stage creates shared containers used by the later model sections.

- `completed_results` stores one row per completed or skipped model run.
- `prediction_tables` stores per-example predictions for error analysis.
- `trained_models` stores reusable fitted model objects when available.

Governance purpose: every model section appends to the same result structure, so the final comparison table is generated from actual run outputs rather than manually entered values.


In [ ]:
completed_results = []
prediction_tables = {}
trained_models = {}

## 5. Dummy Baselines

This stage evaluates non-learning baselines. These baselines are important because they establish a minimum reference point before interpreting more complex models.

Baselines used:

| Model | Behavior | Why it matters |
|---|---|---|
| `random_uniform` | Samples uniformly from the selected label IDs. | Tests performance expected from chance under equal class probability. |
| `random_train_distribution` | Samples labels according to the training label distribution. | Reflects class imbalance without learning from text. |
| `majority_baseline` | Always predicts the most frequent training label. | Provides a strong imbalance-aware dummy baseline for accuracy comparison. |

Evaluation metrics saved for each baseline:

- accuracy
- macro-F1
- weighted-F1
- per-class precision/recall/F1 via classification report
- confusion matrix

Macro-F1 is the primary governance metric because it penalises poor performance on minority classes more clearly than accuracy.


In [ ]:
baseline_results, baseline_prediction_tables = run_baseline_experiments(
    train_df,
    test_df,
    id2label,
    paths.results_dir,
    dataset_name=DATASET_NAME,
    seed=SEED,
)
completed_results.extend(baseline_results)
prediction_tables.update(baseline_prediction_tables)
if baseline_results:
    display(pd.DataFrame(baseline_results)[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

,model_name,accuracy,macro_f1,weighted_f1,notes
0,random_uniform,0.049451,0.045640,0.052960,Uniform random over selected labels.
1,random_train_distribution,0.059383,0.044516,0.060032,Random predictions sampled from the training l...
2,majority_baseline,0.120034,0.010717,0.025728,Always predicts the most frequent training label.


## 6. Classical TF-IDF Models

This stage trains sparse-text supervised models using TF-IDF features. These models are fast, interpretable at the feature level, and provide strong non-neural baselines for legal text classification.

Feature extraction hyperparameters:

| Hyperparameter | Values |
|---|---|
| `max_features` | `10000`, `30000` |
| `ngram_range` | unigram `(1, 1)`, unigram+bigram `(1, 2)` |
| `lowercase` | `True` |
| `stop_words` | `None` |

Model configurations:

| Model | Key settings |
|---|---|
| Logistic Regression | `max_iter=2000`; `random_state=42`; tests `class_weight=None` and `class_weight="balanced"` |
| Linear SVM | `LinearSVC`; `random_state=42`; tests `class_weight=None` and `class_weight="balanced"` |
| Multinomial Naive Bayes | Optional comparison model using TF-IDF inputs |

Selection protocol:

1. Train each configuration on the LEDGAR training split.
2. Select the best configuration using validation macro-F1.
3. Evaluate selected models on the test split once.
4. Save the best classical pipeline and vectorizer artifacts separately.

Explainability note: TF-IDF models are useful for coursework governance because their decisions are linked to sparse lexical features rather than hidden contextual embeddings.


In [ ]:
best_classical_model = None
best_classical_name = None

if not RUN_CLASSICAL_MODELS:
    print("Classical models skipped because RUN_CLASSICAL_MODELS is False.")
else:
    classical_output = run_classical_experiments(
        train_df,
        validation_df,
        test_df,
        id2label,
        paths.results_dir,
        max_features_list=MAX_FEATURES_LIST,
        ngram_ranges=NGRAM_RANGES,
        dataset_name=DATASET_NAME,
        seed=SEED,
        run_naive_bayes=RUN_NAIVE_BAYES,
    )
    completed_results.extend(classical_output["results"])
    prediction_tables.update(classical_output["prediction_tables"])
    best_classical_model = classical_output["best_model"]
    best_classical_name = classical_output["best_model_name"]
    if best_classical_model is not None:
        trained_models["best_classical"] = best_classical_model
    if classical_output["results"]:
        display(pd.DataFrame(classical_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

,model_name,accuracy,macro_f1,weighted_f1,notes
0,logistic_regression,0.954987,0.941931,0.954893,Selected on validation macro-F1. Config={'mode...
1,linear_svm,0.963863,0.953366,0.963473,Selected on validation macro-F1. Config={'mode...
2,multinomial_nb,0.932587,0.911258,0.930669,Selected on validation macro-F1. Config={'mode...


## 7. Fine-Tuned Transformer Classifier

This stage optionally fine-tunes a Hugging Face sequence-classification transformer on LEDGAR. The default model is `distilbert-base-uncased` because it is smaller and more practical for coursework hardware than full BERT-size alternatives.

Training settings used by the module:

| Setting | Value |
|---|---:|
| Model | `distilbert-base-uncased` |
| Maximum sequence length | `256` tokens |
| Learning rate | `2e-5` |
| Epochs | `3` |
| Weight decay | `0.01` |
| Batch size | `16` on larger GPUs, otherwise `8` |
| Mixed precision | `fp16=True` when CUDA is available |
| Model selection | best validation `macro_f1` |
| Early stopping | patience `1` when the callback is available |

Runtime governance:

- This section skips gracefully if CUDA/GPU is unavailable.
- Memory or environment failures are caught and recorded as skipped results.
- The transformer is evaluated on the same LEDGAR test labels as the classical models.

Explainability limitation: transformer representations are contextual but less directly inspectable than TF-IDF features, so confusion matrices and misclassified examples are important for interpreting behavior.


In [ ]:
transformer_output = train_transformer_classifier(
    train_df,
    validation_df,
    test_df,
    id2label,
    paths.results_dir,
    model_name=TRANSFORMER_MODEL_NAME,
    max_length=MAX_TRANSFORMER_LENGTH,
    dataset_name=DATASET_NAME,
    seed=SEED,
    run_transformer=RUN_TRANSFORMER,
)
if transformer_output["result"] is not None:
    completed_results.append(transformer_output["result"])
    prediction_tables[TRANSFORMER_MODEL_NAME] = transformer_output["predictions"]
    trained_models["transformer_trainer"] = transformer_output["trainer"]
    display(pd.DataFrame([transformer_output["result"]])[["model_name", "accuracy", "macro_f1", "weighted_f1"]])
elif transformer_output["skip_result"] is not None:
    completed_results.append(transformer_output["skip_result"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,model_name,accuracy,macro_f1,weighted_f1
0,distilbert-base-uncased,0.964497,0.953737,0.964325


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 8. Qwen2.5-Instruct Prompting Baseline

This stage optionally evaluates an instruction-tuned language model as a prompting baseline. Qwen is not fine-tuned; it is only prompted to classify clauses into the fixed LEDGAR label set.

Prompting setup:

| Mode | Description |
|---|---|
| Zero-shot | Provides the clause text and full list of allowed labels. |
| Few-shot | Adds one training example per label when available; validation and test examples are never used as demonstrations. |

Generation and parsing controls:

| Setting | Value |
|---|---:|
| Model | `Qwen/Qwen2.5-3B-Instruct` |
| Evaluation sample | up to `200` test examples, sampled with `SEED=42` |
| Decoding | deterministic, `do_sample=False` |
| New tokens | `max_new_tokens=24` |
| Output requirement | return exactly one allowed label |
| Fuzzy matching | only used when unambiguous, cutoff `0.80` |
| Invalid outputs | marked as `INVALID_PREDICTION` and reported separately |

Governance note: this is not directly equivalent to supervised fine-tuning. The prompted model has different pretraining and task setup, so results should be interpreted as a separate baseline rather than a perfectly fair model-family comparison.


In [ ]:
qwen_output = run_qwen_baseline(
    train_df,
    test_df,
    label2id,
    id2label,
    paths.results_dir,
    model_name=QWEN_MODEL_NAME,
    label_names=label_names,
    eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    dataset_name=DATASET_NAME,
    seed=SEED,
    run_qwen=RUN_QWEN_BASELINE,
)
completed_results.extend(qwen_output["results"])
qwen_predictions_df = qwen_output["predictions"]
qwen_invalid_outputs_df = qwen_output["invalid_outputs"]
qwen_model = qwen_output["model"]
qwen_tokenizer = qwen_output["tokenizer"]
if qwen_output["results"]:
    display(pd.DataFrame(qwen_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,model_name,accuracy,macro_f1,weighted_f1,notes
0,qwen_zero_shot,0.565,0.503949,0.568809,Qwen prompting baseline. Invalid prediction ra...
1,qwen_few_shot,0.110,0.009910,0.021802,Qwen prompting baseline. Invalid prediction ra...


## 9. Small Agentic Review Prototype

This stage demonstrates a small human-in-the-loop clause triage workflow. It is inspired by tool-use/ReAct-style ideas, but it is not a large autonomous agent and it does not provide legal advice.

Workflow:

1. Use the best available supervised classifier to predict a clause type.
2. Estimate prediction confidence where the model supports it.
3. Flag examples below the review threshold as requiring human review.
4. Optionally ask Qwen for a short triage explanation when Qwen loaded successfully.

Prototype settings:

| Setting | Value |
|---|---:|
| Sample size | `20` test examples |
| Review threshold | `0.55` confidence |
| Logistic Regression confidence | maximum predicted probability |
| Linear SVM confidence | transformed decision-function margin |
| Required disclaimer | `This output is for clause triage and research purposes only.` |

Governance limitation: this section is illustrative. It should not be treated as a production legal review system, risk model, or legal advisor.


In [ ]:
agentic_examples_df = run_agentic_review(
    test_df,
    id2label,
    paths.results_dir,
    best_model=best_classical_model,
    qwen_model=qwen_model,
    qwen_tokenizer=qwen_tokenizer,
    run_agentic=RUN_AGENTIC_EXTENSION,
    seed=SEED,
)
if not agentic_examples_df.empty:
    display(agentic_examples_df.head(10))

,text,true_label,predicted_label,confidence,requires_human_review,triage_note,optional_qwen_explanation
0,"Each party hereto shall do and perform, or cau...",Further Assurances,Further Assurances,0.760225,False,This output is for clause triage and research ...,
1,"There is no pending or threatened notice, clai...",Litigations,Litigations,0.724625,False,This output is for clause triage and research ...,
2,The term of this Agreement shall commence on J...,Terms,Terms,0.697402,False,This output is for clause triage and research ...,
3,"The Executive acknowledges that, by reason of ...",Assignments,Assignments,0.648360,False,This output is for clause triage and research ...,
4,"Pledgor will, from time to time, timely pay an...",Taxes,Taxes,0.694286,False,This output is for clause triage and research ...,
5,Except as set forth on Schedule 3.6 as of the ...,Litigations,Litigations,0.663922,False,This output is for clause triage and research ...,
6,This Warrant shall be governed by and construe...,Governing Laws,Governing Laws,0.675921,False,This output is for clause triage and research ...,
7,This Assignment constitutes the entire and fin...,Entire Agreements,Entire Agreements,0.679556,False,This output is for clause triage and research ...,
8,"The term of this Sublease (""Term"") shall comme...",Terms,Terms,0.565141,False,This output is for clause triage and research ...,
9,This Amendment may be executed by the parties ...,Counterparts,Counterparts,0.719196,False,This output is for clause triage and research ...,


## 10. Final Model Comparison

This stage consolidates all completed and skipped model runs into one comparison table. It does not insert or fabricate any metrics; it only formats rows produced by earlier sections.

Comparison columns:

| Column | Meaning |
|---|---|
| `model_family` | baseline, classical, transformer, or prompting family. |
| `model_name` | specific model/configuration name. |
| `training_type` | dummy, supervised, fine-tuned, prompted, or skipped. |
| `dataset` | evaluation dataset, here LEDGAR for the main experiment. |
| `eval_split` | split used for reported metrics, usually test. |
| `sample_size` | number of evaluated examples. |
| `accuracy` | overall exact-label accuracy. |
| `macro_f1` | unweighted mean F1 across classes; primary metric. |
| `weighted_f1` | class-frequency-weighted F1. |
| `notes` | skip reason or relevant run detail. |

The table and macro-F1 plot are saved under `results/` for later inspection.


In [ ]:
comparison_df = save_final_comparison(completed_results, paths.results_dir)
print(f"Saved final comparison to: {paths.results_dir / 'final_model_comparison.csv'}")
display(comparison_df)

Saved final comparison to: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/results/final_model_comparison.csv


,model_family,model_name,training_type,dataset,eval_split,sample_size,accuracy,macro_f1,weighted_f1,invalid_prediction_rate,notes,classification_report_path,confusion_matrix_path
0,baseline,random_uniform,dummy,LEDGAR,test,4732,0.049451,0.045640,0.052960,NaN,Uniform random over selected labels.,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
1,baseline,random_train_distribution,dummy,LEDGAR,test,4732,0.059383,0.044516,0.060032,NaN,Random predictions sampled from the training l...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
2,baseline,majority_baseline,dummy,LEDGAR,test,4732,0.120034,0.010717,0.025728,NaN,Always predicts the most frequent training label.,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
3,classical,logistic_regression,supervised,LEDGAR,test,4732,0.954987,0.941931,0.954893,NaN,Selected on validation macro-F1. Config={'mode...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
4,classical,linear_svm,supervised,LEDGAR,test,4732,0.963863,0.953366,0.963473,NaN,Selected on validation macro-F1. Config={'mode...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
5,classical,multinomial_nb,supervised,LEDGAR,test,4732,0.932587,0.911258,0.930669,NaN,Selected on validation macro-F1. Config={'mode...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
6,transformer,distilbert-base-uncased,fine-tuned supervised,LEDGAR,test,4732,0.964497,0.953737,0.964325,NaN,Fine-tuned Hugging Face sequence classifier.,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
7,llm_prompting,qwen_zero_shot,zero_shot,LEDGAR,test,200,0.565000,0.503949,0.568809,0.045,Qwen prompting baseline. Invalid prediction ra...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
8,llm_prompting,qwen_few_shot,few_shot,LEDGAR,test,200,0.110000,0.009910,0.021802,1.000,Qwen prompting baseline. Invalid prediction ra...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...


## 11. Error Analysis

This stage supports explainability by looking beyond aggregate metrics. It uses saved prediction tables and confusion matrices from earlier stages.

Analyses included:

- top confused label pairs from the best available classical model
- misclassified examples from the best classical model
- transformer misclassified examples when the transformer ran
- class imbalance summary from the processed training data
- Qwen invalid outputs and semantically plausible but non-matching outputs when available

Interpretation focus:

| Issue | Why it matters |
|---|---|
| Class imbalance | Accuracy can look high while minority classes perform poorly. |
| Label ambiguity | Legal clauses may plausibly fit more than one clause type. |
| Long clauses | Transformer truncation and TF-IDF sparsity can affect predictions. |
| Boilerplate wording | Repeated legal phrasing can make labels harder to separate. |
| Invalid LLM outputs | Prompted models may ignore the closed label set. |

This section provides evidence for model behavior without writing final conclusions or inventing results.


In [ ]:
error_outputs = run_error_analysis(
    comparison_df,
    prediction_tables,
    train_df,
    paths.results_dir,
    best_classical_name=best_classical_name,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
)

print(f"Best completed model by macro-F1: {error_outputs.get('best_model_name')}")
if "classical_confusions" in error_outputs:
    print("Top classical confused label pairs:")
    display(error_outputs["classical_confusions"])
if "classical_misclassified" in error_outputs:
    print("Classical misclassified examples:")
    display(error_outputs["classical_misclassified"][["text", "label", "predicted_label"]])
if "transformer_misclassified" in error_outputs:
    print("Transformer misclassified examples:")
    display(error_outputs["transformer_misclassified"][["text", "label", "predicted_label"]])
if "qwen_invalid_outputs" in error_outputs:
    print("Qwen invalid outputs:")
    display(error_outputs["qwen_invalid_outputs"].head(10))
if "qwen_plausible_nonmatching" in error_outputs:
    print("Qwen semantically plausible but non-matching label examples require manual inspection:")
    display(error_outputs["qwen_plausible_nonmatching"].head(10))

print("Class imbalance summary:")
display(error_outputs.get("class_imbalance", pd.DataFrame()).head(20))

Best completed model by macro-F1: distilbert-base-uncased
Top classical confused label pairs:


,label,predicted_label,count
76,Terms,General,9
36,General,Terminations,8
35,General,Taxes,6
25,General,Assignments,6
37,General,Terms,6
54,Notices,General,5
71,Terminations,Terms,5
1,Amendments,Entire Agreements,5
38,General,Waivers,4
79,Terms,Terminations,4


Classical misclassified examples:


,text,label,predicted_label
38,Upon consummation of an IPO or a Sale of the C...,Terminations,Survival
47,Except as otherwise provided below or in the A...,General,Terms
91,Any Property Costs that are not reflected in t...,Expenses,Taxes
108,Perry acknowledges that the terms of this Agre...,Compliance With Laws,Waivers
115,Subject to the provisions set forth in Article...,General,Indemnifications
182,Except as shall otherwise be stated herein or ...,General,Terms
200,Each Subsidiary of the Company that is or beco...,Notices,General
228,Upon the Completion Date with respect to the F...,Further Assurances,Terminations
240,"The costs and expenses (including, but not lim...",Expenses,Indemnifications
356,Notices and all other communications contempla...,General,Notices


Transformer misclassified examples:


...

,mode,text,label,label_id,raw_output,predicted_label,predicted_label_id,is_invalid
32,zero_shot,The Award of Performance Shares (as set forth ...,Terms,9,Governing Laws\nExplanation: The clause speci...,INVALID_PREDICTION,<NA>,True
42,zero_shot,"Each Party shall deliver all notices, requests...",Notices,1,"Notions\nThe predicted label ""Notions"" does n...",INVALID_PREDICTION,<NA>,True
60,zero_shot,"All demands, notices and communications hereun...",Notices,1,"Notions\nThe predicted label ""Notions"" does n...",INVALID_PREDICTION,<NA>,True
72,zero_shot,Any notice or document required to be filed wi...,Notices,1,General\nThe predicted label for the given cl...,INVALID_PREDICTION,<NA>,True
86,zero_shot,"The Borrower will, and will cause each Princip...",Taxes,12,Compliance With Laws\nExplanation: The clause...,INVALID_PREDICTION,<NA>,True
171,zero_shot,The Guarantor shall pay on demand all out-of-p...,Expenses,8,Compliance With Laws\nThis prediction seems i...,INVALID_PREDICTION,<NA>,True
174,zero_shot,Any notice to the Company provided for in this...,Notices,1,"Notions\nThe predicted label ""Notions"" does n...",INVALID_PREDICTION,<NA>,True
198,zero_shot,The Consolidated Parties have filed all Federa...,Taxes,12,Compliance With Laws\nExplanation: The clause...,INVALID_PREDICTION,<NA>,True
199,zero_shot,"This Confirmation is effective as of April 1, ...",Terms,9,Governing Laws\nThe predicted label does not ...,INVALID_PREDICTION,<NA>,True
200,few_shot,"Each party hereto shall do and perform, or cau...",Further Assurances,14,"such actions, as may be necessary or desirabl...",INVALID_PREDICTION,<NA>,True


Qwen semantically plausible but non-matching label examples require manual inspection:


,mode,text,label,label_id,raw_output,predicted_label,predicted_label_id,is_invalid
1,zero_shot,"There is no pending or threatened notice, clai...",Litigations,13,Compliance With Laws,Compliance With Laws,17,False
2,zero_shot,The term of this Agreement shall commence on J...,Terms,9,Compliance With Laws,Compliance With Laws,17,False
3,zero_shot,"The Executive acknowledges that, by reason of ...",Assignments,7,Compliance With Laws\nThis clause discusses o...,Compliance With Laws,17,False
5,zero_shot,Except as set forth on Schedule 3.6 as of the ...,Litigations,13,Compliance With Laws,Compliance With Laws,17,False
7,zero_shot,This Assignment constitutes the entire and fin...,Entire Agreements,3,Amendments,Amendments,5,False
12,zero_shot,"Borrower shall, and shall cause each Credit Pa...",Insurances,11,Compliance With Laws,Compliance With Laws,17,False
14,zero_shot,To request the issuance of a Letter of Credit ...,Amendments,5,Compliance With Laws,Compliance With Laws,17,False
19,zero_shot,"Subject to the other terms of this Agreement, ...",Further Assurances,14,General,General,16,False
20,zero_shot,Each Credit Party executing this Agreement ack...,Terminations,10,Compliance With Laws,Compliance With Laws,17,False
23,zero_shot,"As between the First Lien Secured Parties, the...",Insurances,11,Compliance With Laws,Compliance With Laws,17,False


Class imbalance summary:


,label,train_count
0,Governing Laws,3136
1,Notices,2445
2,Counterparts,2376
3,Entire Agreements,2318
4,Severability,1774
5,Amendments,1460
6,Survival,1442
7,Assignments,1308
8,Expenses,1211
9,Terms,1142


## 12. Report Artifact Exports

This stage writes the report-facing CSV tables, copied/renamed figures, Qwen prompt/prediction tables, and runtime environment metadata needed to fill the LaTeX TODOs. It does not edit the report or invent missing metrics. If transformer or Qwen sections were skipped, their rows remain marked as skipped in the exported tables.


In [ ]:
from modules.report_exports import export_report_artifacts

report_artifacts = export_report_artifacts(
    paths=paths,
    processed_splits=processed_splits,
    label2id=label2id,
    id2label=id2label,
    completed_results=completed_results,
    prediction_tables=prediction_tables,
    error_outputs=error_outputs,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
    seed=SEED,
    dataset_name=DATASET_NAME,
    max_features_list=MAX_FEATURES_LIST,
    ngram_ranges=NGRAM_RANGES,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    max_transformer_length=MAX_TRANSFORMER_LENGTH,
    qwen_model_name=QWEN_MODEL_NAME,
    qwen_eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    qwen_few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    run_naive_bayes=RUN_NAIVE_BAYES,
)

print("Saved report artifacts:")
for artifact_name, artifact_path in report_artifacts.items():
    print(f"- {artifact_name}: {artifact_path}")


Saved report artifacts:
- data_summary: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/data_summary.csv
- label_distribution: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/label_distribution.csv
- main_results: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/main_results.csv
- predictions: {'distilbert-base-uncased': PosixPath('/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/predictions/distilbert_base_uncased_test_predictions.jsonl'), 'linear_svm': PosixPath('/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/predictions/linear_svm_test_predictions.jsonl'), 'logistic_regression': PosixPath('/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/predictions/logistic_regression_test_predictions.jsonl'), 'majority_baseline': PosixPath('/content/drive/My

## 13. Method Notes and Limitations

These notes summarise the modelling design choices for transparency. They are methodological notes, not final coursework conclusions.

**Aim.** Compare dummy baselines, classical supervised models, an optional fine-tuned transformer, and an optional instruction-tuned LLM prompting baseline for LEDGAR legal clause classification.

**Dataset.** LEDGAR provides legal clause/provision texts with clause type labels and official train/validation/test splits. CUAD is downloaded separately because it has a span-extraction contract review format and is not merged into LEDGAR.

**Preprocessing.** The pipeline standardises columns, normalises whitespace only, removes empty examples and exact duplicate text-label pairs, selects the top-k labels using the training split, and preserves official splits.

**Baselines.** Random and majority baselines establish lower-bound performance for the selected label set.

**Classical models.** TF-IDF Logistic Regression, Linear SVM, and optional Naive Bayes provide efficient sparse-text baselines with transparent feature extraction settings.

**Transformer.** DistilBERT or LegalBERT tests whether contextual representations improve clause classification when fine-tuned on LEDGAR.

**Qwen.** Qwen2.5-Instruct is used as a zero-shot/few-shot prompting baseline to test whether an instruction-tuned LLM can classify clauses without task-specific fine-tuning.

**Agentic extension.** The prototype demonstrates a human-in-the-loop clause triage workflow using classifier confidence and optional LLM explanation. This output is for clause triage and research purposes only, not legal advice.

**Limitations to discuss after results are generated.** LEDGAR labels are clause types, not legal risk labels. Class imbalance affects macro-F1. Prompted LLM performance may be sensitive to prompt wording. Fine-tuned classifiers and prompted LLMs are not perfectly comparable because their training/setup differs. The agentic workflow is illustrative, not production-ready.
